In [ ]:
%matplotlib inline
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import timeit as tt

In [ ]:
weights = np.array( [[1,2,3,4],[5,6,7,8],[9,10,11,12]], dtype=np.float32)
inputs = np.array([[0.1,0.2,0.3]], dtype=np.float32)
outputs = np.dot(inputs, weights)

print ("Inputs (Shape):\n", inputs.shape)
print ("Output (Shape):\n", outputs.shape)
print ("Weights (Shape):\n", weights.shape)

print ("Inputs:\n", inputs)
print ("Weights:\n", weights)
print ("Output:\n", outputs)

print ()
print ('Input\t\t\t Weights\t\t\t  Output')
print ( inputs[0], '   . \t', weights[0], '\t\t= ', outputs[0])
for i in range(1,3): print ('\t\t\t', weights[i])


Inputs (Shape):
 (1, 3)
Output (Shape):
 (1, 4)
Weights (Shape):
 (3, 4)
Inputs:
 [[0.1 0.2 0.3]]
Weights:
 [[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]]
Output:
 [[3.8000002 4.4       5.        5.6000004]]

Input			 Weights			  Output
[0.1 0.2 0.3]    . 	 [1. 2. 3. 4.] 		=  [3.8000002 4.4       5.        5.6000004]
			 [5. 6. 7. 8.]
			 [ 9. 10. 11. 12.]


In [ ]:
# how its done in dot.sv
def pydot(inputs,weights):
    inputs = inputs[0] # remove outer nesting
    outs = np.zeros(weights.shape[1], dtype=np.float32)
    for i in range(weights.shape[0]): # input length
        for j in range(weights.shape[1]): # output length
            outs[j] = outs[j] + weights[i][j] * inputs[i]
    return outs

# my results
print (pydot(inputs,weights))
# reference results
print (np.dot(inputs, weights)[0])

[3.8000002 4.4       5.        5.6000004]
[3.8000002 4.4       5.        5.6000004]


In [ ]:
from pynq import Overlay
from pynq import MMIO
from pynq import allocate

class HwDot():
    def __init__(self, bitstream):
        self.overlay = Overlay(bitstream)

        self.dma20x10 = self.overlay.axi_dma_0
        self.dma40x20 = self.overlay.axi_dma_1
        self.dma80x40 = self.overlay.axi_dma_2

        self.in20x10 = allocate(shape=(20,), dtype=np.float32)
        self.out20x10 = allocate(shape=(10,), dtype=np.float32)

        self.in40x20 = allocate(shape=(40,), dtype=np.float32)
        self.out40x20 = allocate(shape=(20,), dtype=np.float32)

        self.in80x40 = allocate(shape=(80,), dtype=np.float32)
        self.out80x40 = allocate(shape=(40,), dtype=np.float32)

    def dot20x10(self,inputs):
        np.copyto(self.in20x10, inputs)
        return self._dot(self.dma20x10, self.in20x10, self.out20x10)

    def dot40x20(self,inputs):
        np.copyto(self.in40x20, inputs)
        return self._dot(self.dma40x20, self.in40x20, self.out40x20)

    def dot80x40(self,inputs):
        np.copyto(self.in80x40, inputs)
        return self._dot(self.dma80x40,self.in80x40, self.out80x40)

    def _dot(self, dma, inputs, outputs):

        dma.sendchannel.transfer(inputs)
        dma.recvchannel.transfer(outputs)

        dma.sendchannel.wait()
        dma.recvchannel.wait()

        return outputs

ModuleNotFoundError: No module named 'pynq'

In [ ]:
def approx_equal( v0, v1, error = 1E-5):
    results = []
    for (x, y) in zip (v0, v1):
        if (abs(x-y) < error):
            results.append(True)
        else: results.append(False)
    return results

In [ ]:
with open('weights.20x10.json') as f:
    weights20x10 = np.array(json.load(f))
with open('inputs.20x10.json') as f:
    inputs20x10 = json.load(f)

# software
sw_outputs = np.dot( [inputs20x10], weights20x10)

unpipe_dot = HwDot('unpipelined.bit')
unpipe_outputs = unpipe_dot.dot20x10(inputs20x10)

equal = approx_equal(sw_outputs[0], unpipe_outputs)

print ('Equal: ', all(equal))

def py_test():  return pydot( [inputs20x10], weights20x10)
def np_test():  return np.dot( [inputs20x10], weights20x10)
def unpipe_test(): return unpipe_dot.dot20x10(inputs20x10)

print("Timing Python")
time = tt.timeit(py_test, number=1000)
print("Total Time:" + str(time) + " seconds")
print()

print("Timing Numpy")
time = tt.timeit(np_test, number=1000)
print("Total Time:" + str(time) + " seconds")
print()

print("Timing Unpipelined Hardware")
time = tt.timeit(unpipe_test, number=1000)
print("Total Time:" + str(time) + " seconds")
print()

FileNotFoundError: [Errno 2] No such file or directory: 'weights.20x10.json'

## Update your Bitstream with a Pipelined Dot, then run this block

In [ ]:
with open('weights.20x10.json') as f:
    weights20x10 = np.array(json.load(f))
with open('inputs.20x10.json') as f:
    inputs20x10 = json.load(f)

# software
sw_outputs = np.dot( [inputs20x10], weights20x10)

pipe_dot = HwDot('bitstream.bit')
pipe_outputs = pipe_dot.dot20x10(inputs20x10)

equal = approx_equal(sw_outputs[0], pipe_outputs)

print ('Equal: ', all(equal))

def py_test():  return pydot( [inputs20x10], weights20x10)
def np_test():  return np.dot( [inputs20x10], weights20x10)
def pipe_test(): return pipe_dot.dot20x10(inputs20x10)

print("Timing Python")
time = tt.timeit(py_test, number=1000)
print("Total Time:" + str(time) + " seconds")
print()

print("Timing Numpy")
time = tt.timeit(np_test, number=1000)
print("Total Time:" + str(time) + " seconds")
print()

print("Timing Unpipelined Hardware")
time = tt.timeit(pipe_test, number=1000)
print("Total Time:" + str(time) + " seconds")
print()

## Now update this to time the 20x10, 40x20, and 80x40 Dots.  

## Then you need to estimate how the Hardware compares to NumPy.  And estimate when the Hardware will be faster than NumPy.  

In [ ]:
#load weights & inputs
with open('weights.20x10.json') as f:
    weights20x10 = np.array(json.load(f))
with open('inputs.20x10.json') as f:
    inputs20x10 = json.load(f)

with open('weights.40x20.json') as f:
    weights40x20 = np.array(json.load(f))
with open('inputs.40x20.json') as f:
    inputs40x20 = json.load(f)

with open('weights.80x40.json') as f:
    weights80x40 = np.array(json.load(f))
with open('inputs.80x40.json') as f:
    inputs80x40 = json.load(f)

In [ ]:
print("20x10 Correct:",
      all(approx_equal(np.dot([inputs20x10], weights20x10)[0],
                       pipe_dot.dot20x10(inputs20x10))))

print("40x20 Correct:",
      all(approx_equal(np.dot([inputs40x20], weights40x20)[0],
                       pipe_dot.dot40x20(inputs40x20))))

print("80x40 Correct:",
      all(approx_equal(np.dot([inputs80x40], weights80x40)[0],
                       pipe_dot.dot80x40(inputs80x40))))

In [ ]:
#timing functions
def py_20x10():  return pydot([inputs20x10], weights20x10)
def np_20x10():  return np.dot([inputs20x10], weights20x10)
def hw_20x10():  return pipe_dot.dot20x10(inputs20x10)

def py_40x20():  return pydot([inputs40x20], weights40x20)
def np_40x20():  return np.dot([inputs40x20], weights40x20)
def hw_40x20():  return pipe_dot.dot40x20(inputs40x20)

def py_80x40():  return pydot([inputs80x40], weights80x40)
def np_80x40():  return np.dot([inputs80x40], weights80x40)
def hw_80x40():  return pipe_dot.dot80x40(inputs80x40)

In [ ]:
#run time functions
def run_timing(label, py_fn, np_fn, hw_fn):
    print(label)
    print("Python:", tt.timeit(py_fn, number=1000), "seconds")
    print("NumPy :", tt.timeit(np_fn, number=1000), "seconds")
    print("HW    :", tt.timeit(hw_fn, number=1000), "seconds")
    print()

run_timing("20x10 Dot", py_20x10, np_20x10, hw_20x10)
run_timing("40x20 Dot", py_40x20, np_40x20, hw_40x20)
run_timing("80x40 Dot", py_80x40, np_80x40, hw_80x40)

**Estimate how the Hardware compares to NumPy:**

For the 3 tested matrix sizes of 20×10, 40×20, and 80×40, NumPy does way better than the hardware accelerator. This is due to numpy executing more optimized vectorized C code on the CPU with minimal overhead, while the hardware version has additional costs from DMA transfers and AXI stream handshaking. Essentially, as the matrix sizes increase, the hardware version time increases slower than the NumPy one. This tells us that it has better scalability and that the python version scales the worst due to nested loops and interpreter overhead.

**Estimate when the Hardware will be faster than NumPy:**

The hardware accelerator is expected to be faster than NumPy when the dot product size is much larger and multiple dot products are streamed back to back. It can also occur when DMA setup and transfer costs are amortized over many computations and when the pipeline remains fully utilized. Basically, the FPGA can exploit deep pipelining and parallelism, while numpy is limited by memory bandwidth and CPU resources.